# Session 7 Scan 4 V1 Column Functional Correlation Matrix

这个 notebook 使用 `microns_phase3.nda` / DataJoint 访问 functional data，并读取你已经筛选出的 **V1 column 126 个功能配准神经元**；随后选取 `session=7, scan_idx=4` 的这些神经元 activity traces，计算它们组成网络的 Pearson correlation matrix。

输出文件会保存到当前工作目录下的 `outputs_session7scan4_v1_column_126/`。

In [ ]:
from pathlib import Path
import os
import pickle

import numpy as np
import pandas as pd
import datajoint as dj
import matplotlib.pyplot as plt
import seaborn as sns

from microns_phase3 import nda

pd.set_option("display.max_columns", 100)
sns.set_theme(style="white", context="notebook")

SESSION = 7
SCAN_IDX = 4
SCAN_KEY = {"session": SESSION, "scan_idx": SCAN_IDX}

WORKDIR = Path.cwd()
DATA_DIR_CANDIDATES = [
    WORKDIR / "functional data access and analysis",
    WORKDIR / "Functional data access and analysis",
    WORKDIR,
]
OUTDIR = WORKDIR / "outputs_session7scan4_v1_column_126"
OUTDIR.mkdir(exist_ok=True)
EXPECTED_V1_COLUMN_UNITS = 126
SAVE_PAIRWISE_LONG_TABLE = True  # 126 个神经元只有 7,875 对，保存 long table 很轻量。

print("Working directory:", WORKDIR)
print("Output directory:", OUTDIR)
print("DataJoint config host:", dj.config.get("database.host"))

## 1. 确认 DataJoint 中的 scan 和 unit 数量

In [ ]:
scan_rel = nda.Scan & SCAN_KEY
unit_rel = nda.ScanUnit & SCAN_KEY
activity_rel = nda.Activity & SCAN_KEY

print("Scan rows:", len(scan_rel))
print("ScanUnit rows:", len(unit_rel))
print("Activity rows:", len(activity_rel))
display(scan_rel)

## 2. 读取已经筛选出的 V1 column 神经元数据

代码会优先在 `functional data access and analysis/` 文件夹中搜索 `.pkl/.csv/.parquet/.feather/.txt` 表格，文件名中带有 `v1`、`column`、`126` 的候选会排在最前。

理想输入表包含 `session`、`scan_idx`、`unit_id`。如果你的文件只有 126 个 `unit_id`，notebook 会默认这些 unit 属于 `session=7, scan_idx=4`。为了避免误用全量 coregistration 数据，这里不会自动退回到 CAVE 的 5625 个 matched units。

In [ ]:
def _load_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix in {".pkl", ".pickle"}:
        with open(path, "rb") as f:
            obj = pickle.load(f)
        if isinstance(obj, pd.DataFrame):
            return obj
        if isinstance(obj, (list, tuple, set, np.ndarray, pd.Series)):
            return pd.DataFrame({"unit_id": list(obj)})
        if isinstance(obj, dict):
            # Common cases: {"unit_id": [...]}, {"unit_ids": [...]}, or a dataframe-like dict.
            if "unit_ids" in obj and "unit_id" not in obj:
                return pd.DataFrame({"unit_id": obj["unit_ids"]})
            return pd.DataFrame(obj)
        return pd.DataFrame(obj)
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".feather":
        return pd.read_feather(path)
    if suffix == ".txt":
        values = [line.strip() for line in path.read_text().splitlines() if line.strip()]
        return pd.DataFrame({"unit_id": values})
    raise ValueError(f"Unsupported file type: {path}")


def _normalize_unit_table(df: pd.DataFrame):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    # Accept a few likely unit-id names from manual exports.
    rename_candidates = {
        "unit_ids": "unit_id",
        "unit": "unit_id",
        "functional_unit_id": "unit_id",
        "scan_unit_id": "unit_id",
    }
    for old, new in rename_candidates.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})

    if "unit_id" not in df.columns:
        return None

    if "session" not in df.columns:
        df["session"] = SESSION
    if "scan_idx" not in df.columns:
        df["scan_idx"] = SCAN_IDX

    df = df.dropna(subset=["unit_id"]).copy()
    df["session"] = df["session"].astype(int)
    df["scan_idx"] = df["scan_idx"].astype(int)
    df["unit_id"] = df["unit_id"].astype(int)
    return df


def find_v1_column_unit_table():
    patterns = ["*.pkl", "*.pickle", "*.csv", "*.parquet", "*.feather", "*.txt"]
    candidates = []
    for data_dir in DATA_DIR_CANDIDATES:
        if not data_dir.exists():
            continue
        for pattern in patterns:
            candidates.extend(data_dir.rglob(pattern))

    def score(path: Path) -> int:
        name = path.name.lower()
        parent = str(path.parent).lower()
        text = name + " " + parent
        strong = ["v1", "column", "126", "session7", "scan4", "s7", "scan_4"]
        weak = ["coreg", "filtered", "neuron", "functional", "unit", "max_num_scan", "ext", "exc"]
        return 10 * sum(k in text for k in strong) + sum(k in text for k in weak)

    for path in sorted(set(candidates), key=lambda p: (-score(p), len(str(p)))):
        try:
            raw = _load_table(path)
            df = _normalize_unit_table(raw)
        except Exception as exc:
            print(f"Skip {path}: {exc}")
            continue
        if df is None:
            print(f"Skip {path.name}: missing unit_id column")
            continue
        n_this_scan = df.query("session == @SESSION and scan_idx == @SCAN_IDX")["unit_id"].nunique()
        print(f"Candidate {path}: {n_this_scan} unique units for session={SESSION}, scan_idx={SCAN_IDX}")
        if n_this_scan == EXPECTED_V1_COLUMN_UNITS:
            print(f"Loaded V1 column unit table: {path}")
            return df, path

    return None, None


df_coreg_filtered, source_path = find_v1_column_unit_table()

if df_coreg_filtered is None:
    raise FileNotFoundError(
        f"没有找到包含 {EXPECTED_V1_COLUMN_UNITS} 个 V1 column unit_id 的本地文件。"
        "请把你筛选出的 V1 column 神经元表放到 'functional data access and analysis/' 文件夹，"
        "或放在当前目录；文件至少需要一列 unit_id。"
    )

print("Source:", source_path)
print("Shape:", df_coreg_filtered.shape)
display(df_coreg_filtered.head())

In [ ]:
# Deliberately no CAVE fallback here.
# The CAVE coregistration table contains thousands of matched units for session7scan4;
# this analysis should use only your preselected 126 V1 column neurons.

## 3. 选取 session 7 scan 4 的 126 个 V1 column 神经元

In [ ]:
df_session_scan = (
    df_coreg_filtered
    .query("session == @SESSION and scan_idx == @SCAN_IDX")
    .copy()
)

# Keep one row per functional unit. If the source table contains EM ids or V1 column metadata,
# they are preserved in df_units for downstream merging.
df_units = (
    df_session_scan
    .drop_duplicates(subset=["session", "scan_idx", "unit_id"])
    .sort_values("unit_id")
    .reset_index(drop=True)
)

unit_ids = df_units["unit_id"].astype(int).tolist()
print(f"V1 column units in session {SESSION}, scan {SCAN_IDX}: {len(unit_ids)}")
display(df_units.head())

if len(unit_ids) != EXPECTED_V1_COLUMN_UNITS:
    raise ValueError(
        f"Expected {EXPECTED_V1_COLUMN_UNITS} V1 column units, but found {len(unit_ids)}. "
        "Please check that the loaded file is your manually selected V1 column table."
    )

## 4. 用 DataJoint 读取 activity traces

这里使用 `nda.Activity.trace`，也就是 spike extraction 后的 activity trace。若要改成原始 calcium fluorescence，可通过 `nda.ScanUnit` 关联 `nda.Fluorescence` 获取 `trace`。

In [ ]:
selected_keys = [{"session": SESSION, "scan_idx": SCAN_IDX, "unit_id": int(u)} for u in unit_ids]
selected_activity = nda.Activity & selected_keys

print("Requested units:", len(selected_keys))
print("Activity traces found:", len(selected_activity))

activity_records = selected_activity.fetch("KEY", "trace", order_by="unit_id")
keys, traces = activity_records[0], activity_records[1]
activity_unit_ids = np.array([k["unit_id"] for k in keys], dtype=int)

trace_lengths = np.array([len(np.asarray(t).squeeze()) for t in traces])
print("Trace length summary:")
print(pd.Series(trace_lengths).describe())

min_len = int(trace_lengths.min())
trace_matrix = np.vstack([np.asarray(t).squeeze()[:min_len] for t in traces])
print("Trace matrix shape (units x frames):", trace_matrix.shape)

# Save the exact units used.
used_units = pd.DataFrame(keys).merge(df_units, on=["session", "scan_idx", "unit_id"], how="left")
used_units.to_csv(OUTDIR / "session7_scan4_v1_column_126_units_used.csv", index=False)
display(used_units.head())

## 5. 计算 correlation matrix

In [ ]:
def zscore_rows(x: np.ndarray) -> np.ndarray:
    x = x.astype(float)
    mean = np.nanmean(x, axis=1, keepdims=True)
    std = np.nanstd(x, axis=1, keepdims=True)
    std[std == 0] = np.nan
    return (x - mean) / std

z_traces = zscore_rows(trace_matrix)
valid_rows = np.isfinite(z_traces).all(axis=1)

if not valid_rows.all():
    dropped = activity_unit_ids[~valid_rows]
    print(f"Dropping {len(dropped)} units with non-finite or constant traces:", dropped[:20])

z_traces_valid = z_traces[valid_rows]
valid_unit_ids = activity_unit_ids[valid_rows]

corr = np.corrcoef(z_traces_valid)
corr_df = pd.DataFrame(corr, index=valid_unit_ids, columns=valid_unit_ids)

corr_csv = OUTDIR / "session7_scan4_v1_column_126_activity_correlation_matrix.csv"
corr_npy = OUTDIR / "session7_scan4_v1_column_126_activity_correlation_matrix.npy"
corr_df.to_csv(corr_csv)
np.save(corr_npy, corr)

print("Correlation matrix shape:", corr_df.shape)
print("Saved:", corr_csv)
print("Saved:", corr_npy)
display(corr_df.iloc[:10, :10])

## 6. 可视化 correlation matrix 和相关性分布

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7), dpi=150)
sns.heatmap(
    corr_df,
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    xticklabels=False,
    yticklabels=False,
    square=True,
    cbar_kws={"label": "Pearson r"},
    ax=ax,
)
ax.set_title(f"Session {SESSION} Scan {SCAN_IDX} V1 Column Functional Correlation")
ax.set_xlabel("unit_id")
ax.set_ylabel("unit_id")
fig.tight_layout()
heatmap_path = OUTDIR / "session7_scan4_v1_column_126_activity_correlation_heatmap.png"
fig.savefig(heatmap_path, bbox_inches="tight")
plt.show()
print("Saved:", heatmap_path)

upper = corr[np.triu_indices_from(corr, k=1)]
fig, ax = plt.subplots(figsize=(6, 3.5), dpi=150)
ax.hist(upper[np.isfinite(upper)], bins=60, color="#3f7f93", edgecolor="white")
ax.axvline(np.nanmean(upper), color="black", lw=1.5, label=f"mean={np.nanmean(upper):.3f}")
ax.set_xlabel("Pairwise Pearson r")
ax.set_ylabel("Neuron pairs")
ax.legend(frameon=False)
fig.tight_layout()
hist_path = OUTDIR / "session7_scan4_v1_column_126_activity_correlation_distribution.png"
fig.savefig(hist_path, bbox_inches="tight")
plt.show()
print("Saved:", hist_path)

## 7. 可选：保存长表格式 pairwise correlations

如果神经元数量很多，长表会比较大；默认仍保存，便于后续按 `unit_i/unit_j` 合并其他网络或结构信息。

In [ ]:
if SAVE_PAIRWISE_LONG_TABLE:
    i_idx, j_idx = np.triu_indices_from(corr, k=1)
    pairwise_corr = pd.DataFrame({
        "unit_i": valid_unit_ids[i_idx],
        "unit_j": valid_unit_ids[j_idx],
        "pearson_r": corr[i_idx, j_idx],
    })
    pairwise_csv = OUTDIR / "session7_scan4_v1_column_126_pairwise_activity_correlations.csv"
    pairwise_corr.to_csv(pairwise_csv, index=False)
    print("Pairwise rows:", len(pairwise_corr))
    print("Saved:", pairwise_csv)
    display(pairwise_corr.head())
else:
    n = len(valid_unit_ids)
    print(
        f"Skipped pairwise long-table export for {n} units "
        f"({n * (n - 1) // 2:,} pairs). Set SAVE_PAIRWISE_LONG_TABLE = True to save it."
    )